# marketsim-v2 — run scenarios on Google Colab

Run the crisis-calibrated market simulator in the cloud, with no local setup. This
notebook clones your GitHub repo, installs the package, runs scenario batteries for
Papers 3 / 5 / 6, and saves the result CSVs to your Google Drive (or Supabase).

**How to use:** Runtime -> Run all, or run each cell top to bottom. Cells are grouped:
setup, sanity check, pick a scenario, run a batch or sweep, save results.

_Vercel is not involved and cannot run these batches — this notebook is the supported
way to execute the simulator._


## 1. Setup — clone the repo and install


In [ ]:
# Repo: https://github.com/zalliata/marketsim-v2
# If the repo is PRIVATE, uncomment the token line and paste a GitHub token
# (github.com -> Settings -> Developer settings -> fine-grained token, Contents: read).
REPO = 'https://github.com/zalliata/marketsim-v2.git'
# TOKEN = 'ghp_xxx'; REPO = REPO.replace('https://', f'https://{TOKEN}@')

import os, shutil
if os.path.exists('marketsim-v2'):
    shutil.rmtree('marketsim-v2')
!git clone -q $REPO
print('cloned:', os.path.exists('marketsim-v2'))


In [ ]:
# The package sits at the repo root (pyproject.toml is directly inside marketsim-v2/).
PKG_DIR = 'marketsim-v2'
import os
assert os.path.exists(os.path.join(PKG_DIR, 'pyproject.toml')), \
    f'pyproject.toml not found in {PKG_DIR} — check the repo cloned correctly'
%cd $PKG_DIR
!pip install -q -e .
print('installed')


## 2. Sanity check — run the test suite
Confirms the engine, agents, and scenario registry are healthy before you spend compute.


In [ ]:
!python -m pytest -q


## 3. See what you can run


In [ ]:
!marketsim list                 # all scenarios
# !marketsim list --paper P3    # just one paper's battery
# !marketsim show p3-cost-sweep-adversary   # detail for one scenario


## 4. (Optional) Save results to Google Drive
Mounts your Drive so result CSVs persist after the Colab session ends. Skip this cell
to keep results only in the temporary Colab filesystem (you can still download them).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/marketsim-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results will be saved under', RESULTS_DIR)


## 5. (Optional) Use Supabase instead of local CSVs
Leave this cell unrun to write local CSVs (default). Fill in your project's URL and
service key to write straight to the same Supabase tables the old app used.
In Colab, prefer the key icon (Secrets) over pasting keys into the notebook.


In [ ]:
import os
# os.environ['SUPABASE_URL'] = 'https://xxxx.supabase.co'
# os.environ['SUPABASE_SERVICE_KEY'] = 'your-service-key'
# !pip install -q supabase
print('Supabase configured:' , bool(os.environ.get('SUPABASE_URL')))


## 6. Run a single scenario (quick smoke test)
One iteration, printed summary — good for checking a scenario behaves before a full batch.


In [ ]:
!marketsim run rq2-adversary-enters --seed 0


## 7. Run a Monte Carlo batch
N seeded iterations of one scenario, written to storage. Start small (e.g. 20) to gauge
timing, then scale to the 100 iterations the papers use.


In [ ]:
SCENARIO = 'p3-cost-sweep-adversary'   # change to any id from `marketsim list`
ITERATIONS = 20                        # bump to 100 for the paper runs
!marketsim batch $SCENARIO --iterations $ITERATIONS --storage local


## 8. Run a parameter sweep (Papers 3 / 5 / 6)
Sweeps walk a grid (transaction cost for P3, adversary market share for P5/P6) and run a
full batch at each grid point. These are the heavy jobs — run them here, not on your laptop.


In [ ]:
# Paper 3 — transaction-cost floor sweep (0-50 bps)
!marketsim sweep p3-cost-sweep-adversary --iterations 50

# Paper 5 — detection feasibility (adversary market share 1-50%)
# !marketsim sweep p5-share-sweep-graph --iterations 50

# Paper 6 — algorithmic composition / phase transition
# !marketsim sweep p6-composition-sweep --iterations 25


---
# Full paper batteries

The cells below run **every scenario each new paper needs**, organized by paper, using
the Python API directly (one shared storage backend per paper). Each run is written to
storage (local CSV under `results/`, or Supabase if configured in step 5) and a short
summary is printed.

**Iterations:** the papers use 100 Monte Carlo iterations per condition. That is heavy on
Colab's free tier — start with `ITER = 20` to check timing and outputs, then raise to 100
for the runs you actually report. Sweeps multiply: a 10-point share sweep at 100 iterations
is 1,000 simulations.


### Shared runner
Run this once; the three paper cells below call `run_paper(...)`.


In [ ]:
from marketsim.scenarios import definitions as D
from marketsim.runner.batch import run_batch, run_sweep
from marketsim.storage import make_backend

ITER = 20        # iterations per condition — set to 100 for the reported runs
LLM = 'scripted' # deterministic adversary; use 'anthropic' etc. only for small batches

def run_paper(scenario_ids, iterations=None, llm=None):
    iterations = iterations or ITER
    llm = llm or LLM
    backend = make_backend('auto')   # Supabase if configured (step 5), else local CSV
    try:
        for sid in scenario_ids:
            sc = D.get(sid)
            if sc.sweep:
                stats = run_sweep(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] sweep over {sc.sweep.kind} — {len(stats)} grid points (n={iterations}/point):')
                for st in stats:
                    print(f'    {st.grid_key}={st.grid_value:<8} n={st.n}  '
                          f'vol={st.vol_mean:.4f}  adv_pnl={st.adversary_pnl_mean:>12,.0f}  '
                          f'cascade={st.cascade_freq_mean:.3f}  stab={st.stabilisation_mean:.3f}')
            else:
                st = run_batch(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] batch n={st.n}: vol={st.vol_mean:.4f}  '
                      f'adv_pnl={st.adversary_pnl_mean:,.0f}  cascade={st.cascade_freq_mean:.3f}')
    finally:
        backend.close()
    print('--- done ---')


## Paper 3 — RQ-F4: transaction-cost floors, circuit breakers, deterrence welfare

- `p3-cost-sweep-adversary` — transaction cost 0-50 bps vs coordinated LLM adversaries;
  locates tc* where mean adversary PnL turns negative.
- `p3-cost-sweep-graph` — same sweep vs graph exploiters (your P2 10 bps result is one grid point).
- `p3-circuit-breaker` — intervention arms {none, FTT@tc*, halt} at a common stress trigger,
  for the FTT-vs-halt welfare comparison.

After running, refine the cost grid near the zero-crossing (edit the sweep values in
`marketsim/scenarios/definitions.py`, or run `p3-cost-sweep-adversary` again at 1 bps steps
over the bracket you find).


In [ ]:
PAPER3 = ['p3-cost-sweep-adversary', 'p3-cost-sweep-graph', 'p3-circuit-breaker']
run_paper(PAPER3)          # add iterations=100 for the reported run


## Paper 5 — RQ-F6: detecting coordinated algorithmic manipulation

Two groups:
1. **Labelled battery** for training the detection classifier — three clean baselines and
   six adversary configurations (A1-A3 volatility adversaries, G1-G3 graph exploiters),
   each row tagged manipulated/clean.
2. **Market-share feasibility sweeps** — adversary share 1-50% to find the minimum share at
   which detection becomes viable, for LLM and graph adversaries separately (H2: graph
   strategies detectable at lower share).


In [ ]:
PAPER5_LABELLED = [
    'p5-clean-noise', 'p5-clean-mm', 'p5-clean-defense',   # clean (span the defense configs)
    'p5-manip-A1', 'p5-manip-A2', 'p5-manip-A3',           # volatility adversaries
    'p5-manip-G1', 'p5-manip-G2', 'p5-manip-G3',           # graph exploiters
]
PAPER5_SWEEPS = ['p5-share-sweep-llm', 'p5-share-sweep-graph']

run_paper(PAPER5_LABELLED)   # labelled training data
run_paper(PAPER5_SWEEPS)     # detection-feasibility sweeps


## Paper 6 — RQ-F7: algorithmic concentration, fragility thresholds, phase transition

- `p6-composition-sweep` — adversarial share 5-50% across the defended arena; the
  cascade-frequency and stabilisation columns locate the phase-transition threshold.
- `p6-monoculture` vs `p6-diversity` — single dominant strategy vs a mixed adversary at each
  share level (H2: diversity delays the transition).

These are the heaviest jobs (each sweep runs the battery at every share level). For a first
pass use a smaller `iterations`; split the share grid across sessions if Colab times out.
Feed the resulting stats to `marketsim.metrics.systemic.locate_phase_transition`.


In [ ]:
PAPER6 = ['p6-composition-sweep', 'p6-monoculture', 'p6-diversity']
run_paper(PAPER6, iterations=25)   # raise once you've gauged timing


### Locate the P6 phase transition from a sweep
Example: turn the composition sweep's stabilisation / cascade curves into a threshold estimate.


In [ ]:
from marketsim.metrics.systemic import locate_phase_transition
from marketsim.runner.batch import run_sweep
from marketsim.storage import make_backend

backend = make_backend('local')
try:
    stats = run_sweep(D.get('p6-composition-sweep'), ITER, backend)
finally:
    backend.close()

shares = [s.grid_value for s in stats]
stab   = [s.stabilisation_mean for s in stats]
casc   = [s.cascade_freq_mean for s in stats]
pt = locate_phase_transition(shares, stab, casc)
print('phase transition found:', pt.found, 'at adversary share =', pt.threshold_share)


## 9. Collect the results
Result CSVs land in `results/<run_id>/` (time_steps, scenario_iterations,
agent_snapshots). Copy them to Drive (if mounted) so they survive the session, or zip
and download.


In [ ]:
import glob, shutil, os
runs = sorted(glob.glob('results/*'))
print(f'{len(runs)} run folders:')
for r in runs[:20]: print(' ', r)

# copy to Drive if it was mounted in step 4
if os.path.exists('/content/drive/MyDrive'):
    dest = '/content/drive/MyDrive/marketsim-results'
    os.makedirs(dest, exist_ok=True)
    for r in runs:
        shutil.copytree(r, os.path.join(dest, os.path.basename(r)), dirs_exist_ok=True)
    print('copied to', dest)


In [ ]:
# ...or zip everything and download to your computer
import shutil
shutil.make_archive('marketsim_results', 'zip', 'results')
from google.colab import files
files.download('marketsim_results.zip')


---
### Tips
- **Timing:** run 20 iterations first; multiply to estimate a 100-iteration batch.
- **Reproducibility:** every run is seeded; the same scenario + seed gives identical output.
- **Free-tier limits:** Colab disconnects idle sessions. For the largest P6 sweeps, split
  the share grid across several runs, or use Colab Pro / a cloud VM.
- **Genuine-LLM runs:** add `--llm anthropic` (and set `ANTHROPIC_API_KEY`) to a `run`/`batch`
  command to use a real model instead of the deterministic scripted adversary. Keep batches
  small in that mode — every decision is an API call.
